# 対話：質問・入力・回答・確認を順に見る

授業時間は[演習カード](../activities/day1.md)を確認します。質問欄は空欄から始めます。
送信セルで `SEND=True` にした時だけ有料APIを呼びます。モデル・接続経路・許可枠を明示してください。APIキーは入力欄でのみ受け取り、保存しません。
送信失敗時は停止します。別モデルへの切替・再送は自動で行いません。大学GUIの記録とAPIの記録は経路を区別します。

**スライドとの対応**：37–43「質問の工夫と比較」、85「朝と同じ範囲・同じ3問を読み直す」、86「Methods：手順と評価の条件を確かめる」、89–91（The AI Scientistへの応用）。
今回変えるのは、質問の書き方・原文の範囲・履歴の有無のうち一つです。回答が変わったら、原文へ戻って自分の説明がどう変わったかを記録します。


In [ ]:
# @title 0. 教材を準備する { display-mode: "form" }
from pathlib import Path
import sys
PREPARE_COLAB = False  # @param {type:"boolean"}
RELEASE_COMMIT = "c35b3c0e1532ec91bede4d1547e735607584e80a"
RELEASE_SHA256 = "d105cb72768f2850142da0fdaa6fee5fa9aed34607a396ccbba31863fa7a052d"
BOOTSTRAP_SHA256 = "f1ad167aeec1cf95e2aa835eef73c05fdeae916445985ab69679ec6060d99914"
if PREPARE_COLAB:
    from urllib.request import urlopen
    from hashlib import sha256
    if not RELEASE_COMMIT or not RELEASE_SHA256:
        raise RuntimeError('この改訂版のColab公開準備はまだ完了していません。')
    url = f'https://raw.githubusercontent.com/humansys-lab/multimodal-ai-paper-reading/{RELEASE_COMMIT}/scripts/colab_bootstrap.py'
    with urlopen(url, timeout=40) as response:
        source = response.read(100_001)
    if len(source) > 100_000 or sha256(source).hexdigest() != BOOTSTRAP_SHA256:
        raise RuntimeError('準備コードのハッシュが一致しません。実行を中止しました。')
    setup = {}
    exec(compile(source, 'verified_colab_bootstrap.py', 'exec'), setup)
    ROOT = setup['prepare_notebook'](RELEASE_COMMIT, RELEASE_SHA256, profile='dialogue')
else:
    ROOT = globals().get('ROOT', Path.cwd()).resolve()
    if ROOT.name == 'notebooks': ROOT = ROOT.parent
    if not (ROOT / 'src/seminar_lab/__init__.py').is_file():
        raise RuntimeError('教材フォルダで実行するか、ColabではPREPARE_COLABをTrueにしてください。')
if str(ROOT / 'src') not in sys.path: sys.path.insert(0, str(ROOT / 'src'))
print('教材フォルダ:', ROOT.name)

## 1. この演習で使う道具を読み込む
画像・PDF用のライブラリはこのNotebookでは読み込みません。

In [ ]:
from copy import deepcopy
from seminar_lab.config import load_yaml, material_context
from seminar_lab.client import Session, OpenAITransport, output_text
from seminar_lab.connection import load_connection
from seminar_lab.inputs import text_input
from seminar_lab.ui import activity_description, preview_text, save_pair
from seminar_lab.records import load_records
course = load_yaml(ROOT / 'config/course.yaml')
manifest = load_yaml(ROOT / 'materials/manifest.yaml')

## 2. 教員の接続ファイルを指定する（初回・活動変更時）
教員から配布されたファイルをColab左側のFiles欄へ置き、パスをコピーして`connection_path`へ記入します。学生がモデルの対応表や利用許可を記入する必要はありません。公開の空様式では送信できません。

`L2`＝質問方法の比較、`P1`＝最初のAI読解、`P2a`＝同じ範囲の再読、`METHODS`＝Methodsの読解、`TRANSFER`＝The AI Scientist。共通読解の論文・版・指定範囲・3問は全員同じです。

設定を読み直しても過去の記録と使用回数は保持します。モデルや資料を変えるときは保存してから新規会話にします。金額の利用上限は教員の中継側で管理します。


In [ ]:
from uuid import uuid4
connection_path = ''  # 教員が配布した接続ファイルのパス。キーは書かない
activity_id = 'L2'  # この上の表にある演習名と対応
if 'session' not in globals():
    session = Session()
    permissions = {}  # 同じ許可の再読込みで使用回数をリセットしない
if 'run_id' not in globals():
    run_id = str(uuid4())
runtime = permission = None
if connection_path.strip():
    runtime, permission = load_connection(connection_path, permissions)
    print('モデル:', runtime['model'], '／ 接続経路:', runtime['route'])
else:
    print('接続は未設定です。教員の設定ファイルを指定すると送信を準備できます。')
print(activity_description(course, manifest, activity_id))


## 3. 質問と資料を書く（繰り返し使う）
`question`は自分の問い。`source_excerpt`は送る原文そのもの、`source_location`はページ・節・図です。原文を送らない比較では抜粋を空にします。`new`は履歴なし、`continue`は成功した対話の全履歴を送ります。

送信後は **保存 → 次の実行番号 → このセルで条件変更 → プレビュー → 送信** の順に繰り返します。
### 質問を変える時の見取り図

- **Zero-shot**：例を付けず、目的とほしい形式を伝える。
- **Few-shot**：望む入出力の例を少数、質問の中に示す。モデルの重みを追加学習する操作ではありません。[Brown et al. (2020), §2.1・Fig.2.1](https://arxiv.org/abs/2005.14165)
- **Chain-of-thought**：途中の説明を含む例を示す。[Wei et al. (2022), §2・Fig.1](https://arxiv.org/abs/2201.11903) 例なしで段階的な説明を求める方法は[Zero-shot CoT](https://arxiv.org/abs/2205.11916)。出力された説明から非公開の内部思考を確認できるわけではありません。

まず自分で問いを書き、比較する条件を一つ選びます。短い比較例は[講義の参考文献](../docs/lecture_references.md#授業用の比較例初回読解後に提示)にあります。

複数行の質問や抜粋は、`"""`と`"""`の間に書けます。引用符そのものを含む文章は、囲み方に注意してください。


In [ ]:
question = ''
source_excerpt = ''
source_location = ''
conversation_mode = 'new'
parameters = {'max_output_tokens': 512}
inputs = []  # 入力の準備に失敗したとき、前の入力を残さない
if source_excerpt.strip():
    if not source_location.strip():
        raise ValueError('抜粋の元ページ・節・図を記入してください。')
    inputs = [text_input(source_excerpt, source_location)]

## 4. 実際に送る質問と履歴を見る
少数例を加える時は質問欄に全文を書きます。隠れた指示は加えません。比較は新規会話で同じ条件にします。

In [ ]:
previewed_request = None
if question.strip():
    if runtime is None or permission is None:
        raise ValueError('教員の接続ファイルを先に読み込んでください。')
    context = (globals().get('homework_context') if activity_id == 'HW1' else material_context(course, manifest, activity_id))
    if context is None:
        raise ValueError('Homeworkの論文情報を01の末尾にある切替セルで確認してください。')
    payload = session.preview(runtime, question, inputs, conversation_mode, parameters)
    previewed_request = deepcopy((context, payload, [x.provenance for x in inputs]))
    print('資料と版・指定範囲:', context['material_id'], context.get('source_version'), context.get('assigned_scope'))
    print('実入力の由来:', [x.provenance for x in inputs])
    print(preview_text(payload))
else:
    print('質問欄は空です。自分の問いを入力してください。送信していません。')

## 5. 確認した内容を1回送る（ここで課金）
`SEND=True`で実行し、専用入力欄へ個別キーを入れます。同じ実行番号の再送は停止します。回答が途中・拒否・エラーならその状態を表示し、別モデルへの切替や再送はしません。

In [ ]:
SEND = False
if SEND:
    result = None  # 失敗後に前の回答を今回の結果として扱わない
    if runtime is None or permission is None:
        raise ValueError('教員の接続ファイルを先に読み込んでください。')
    context = (globals().get('homework_context') if activity_id == 'HW1' else material_context(course, manifest, activity_id))
    if context is None:
        raise ValueError('Homeworkの論文情報を01の末尾にある切替セルで確認してください。')
    # キー入力より先に、素材・設定・送信内容を検証
    payload = session.preview(runtime, question, inputs, conversation_mode, parameters)
    if globals().get('previewed_request') != (context, payload, [x.provenance for x in inputs]):
        raise ValueError('質問・入力・活動・設定が変わっています。プレビューを再実行してください。')
    permission.validate(runtime)
    session.check_run_id(run_id)
    from getpass import getpass
    transport = OpenAITransport(runtime, getpass('許可された個別APIキー（表示・保存しません）: '), permission)
    try:
        result = session.run(runtime=runtime, context=context, prompt=question, inputs=inputs,
                             mode=conversation_mode, parameters=parameters, run_id=run_id, transport=transport)
    finally:
        transport.close()
    print('状態:', result['status'])
    if result.get('error_message'):
        print(result['error_message'])
    print(output_text(result))
    print('費用:', result['cost_value'] if result['cost_value'] is not None else '不明')
else:
    print('送信OFF。APIは呼び出していません。')

## 6. 自分の説明を追記し、保存する
`None` は前の記入を保持。JSONLとMarkdownをダウンロードします。

In [ ]:
# Noneは前の記入を保持。文字列を入れた欄だけ更新する。
annotations = {
    'student_explanation': None,  # 自分の説明
    'evidence_location': None,  # 根拠のページ・図・本文
    'student_revision': None,  # AIの回答を確認して直した説明
    'unresolved_point': None,  # まだ分からないこと
    'next_change': None,  # 次に変える質問や入力
    'change_reason': None,  # 変更する理由
}
SAVE = False
if SAVE:
    if not session.records:
        raise ValueError('保存する実行記録がありません。先に対話を実行してください。')
    if (globals().get('result') is None or result.get('run_id') != run_id
            or session.records[-1]['run_id'] != run_id):
        raise ValueError('今回の実行記録がありません。前回の回答へ今回の説明を付けずに停止しました。')
    if any(value is not None and not isinstance(value, str) for value in annotations.values()):
        raise ValueError('記入欄は文字列かNoneにしてください。')
    session.records[-1].update({key: value for key, value in annotations.items() if value is not None})
    paths = save_pair(session.records, ROOT / 'outputs')
    print([p.name for p in paths])
    assert load_records(paths[0]) == session.records
    assert load_records(paths[1]) == session.records
    # Colabのファイル欄からoutputs内の2ファイルをダウンロード。
else:
    print('保存は未実行です。前の記入内容も変更していません。')

## 次の質問
新しい質問の時だけ新しい実行番号を作ります。同じ送信セルの再実行による二重送信は拒否します。

In [ ]:
START_NEXT_RUN = False
if START_NEXT_RUN:
    run_id = str(uuid4())
    print('新しい実行番号を用意しました。質問・新規/継続・プレビューを見直してください。')

## Homeworkへの切替（授業後に使う）
先に共通課題を保存します。以下は、別の原著論文の出典を記録するための空様式です。[論文の選び方](../materials/papers/extensions.md)に沿って出版社ページと本文を自分で確認します。

- 自分で選ぶ場合は、`homework_selection='U_my_paper'`として書誌・版・範囲を記入します。`U_my_paper`は手元の記録の名前で、学籍番号や氏名を使う必要はありません。
- 掲載済みの原著・対象誌・OAを確認した項目だけ、`publication_status='published'`、`article_type='research'`、`open_access=True`、`eligibility_review`を`confirmed`にします。
- AIへ入力できる条件を確認した場合だけ`input_review`を`confirmed`にします。本人による確認を教員の採用や再配布許可とは扱いません。分からなければ送らず相談します。

教員候補を使う場合は教員の素材IDを指定します。切替が成功したら、上の質問・プレビュー・送信・保存のセルを使います。共通課題の記録は残り、Homeworkは別段階になります。


In [ ]:
START_HOMEWORK = False
if START_HOMEWORK:
    homework_context = None
    homework_selection = ''  # 条件を確認したE01等、またはU_<local_id>を選ぶ。P01はHW1の対象外
    local_material = {
        'id': homework_selection,
        'bibliography': {'title': '', 'authors': [], 'journal': ''},  # journalはNatureまたはScience
        'doi': '',
        'source_url': '',  # 出版社の掲載論文ページ（DOIの短縮URLではない）
        'source_version': '',  # 掲載版の日付・確認日
        'assigned_scope': '',  # 読む本文の節・PDFページ・図
        'license': '',  # 確認したライセンスと表示場所
        'publication_status': 'unconfirmed',
        'article_type': 'unconfirmed',
        'open_access': False,
        'eligibility_review': {'status': 'unconfirmed', 'reviewer': '本人'},
        'input_review': {'status': 'unconfirmed', 'reviewer': '本人'},
    }
    homework_context = material_context(course, manifest, 'HW1', homework_selection, local_material)
    # 以前の記録を保持したまま、新規会話とhomework段階へ切り替える
    activity_id = 'HW1'
    run_id = str(uuid4())
    conversation_mode = 'new'
    question = ''
    inputs = []
    print(activity_description(course, manifest, 'HW1'))
    print('記録段階:', homework_context['phase'])